# 1 命令行参数解析

```python
def get_args():
    parser = argparse.ArgumentParser(...)
    # 模型与分词器
    parser.add_argument("--checkpoint_path", required=True, ...)
    parser.add_argument("--vocab_path", required=True, ...)
    parser.add_argument("--merges_path", required=True, ...)
    parser.add_argument("--special_tokens", nargs='*', default=["<|endoftext|>"], ...)
    # 生成超参
    parser.add_argument("--prompt", default="Once upon a time", ...)
    parser.add_argument("--max_new_tokens", type=int, default=256, ...)
    parser.add_argument("--temperature", type=float, default=0.8, ...)
    parser.add_argument("--top_p", type=float, default=0.9, ...)
    # 性能
    parser.add_argument("--device", default="cuda", ...)
    parser.add_argument("--compile", action="store_true", ...)
    return parser.parse_args()
```

* 关键参数：

  * `checkpoint_path`：训练好的 checkpoint 路径，用来恢复模型权重。
  * `vocab_path / merges_path`：BPE 分词器词表与 merges 规则文件。
  * `special_tokens`：特殊符号（如 `<|endoftext|>`）列表。
  * `prompt`：生成的起始提示。
  * `max_new_tokens`：最多生成多少个**新**token。
  * `temperature`：logits 除以 `T` 后再 softmax。`T<1` 更保守，`T>1` 更发散。
  * `top_p`：nucleus 采样阈值，只在累计概率前 `p` 的子集中采样。
  * `device`/`compile`：运行设备/是否用 `torch.compile` 加速。


# 2 核心生成函数

```python
@torch.no_grad()
def generate(model, tokenizer, prompt_ids, max_new_tokens, temperature, top_p):
    model.eval()
    print(tokenizer.decode(prompt_ids.tolist()[0]), end="", flush=True)

    ids = prompt_ids
    for _ in range(max_new_tokens):
        # 截取上下文
        context = ids[:, -model.token_embeddings.weight.shape[1]:]

        # 前向得到 logits，并只取最后一步
        logits = model(context)
        logits = logits[:, -1, :]

        # 温度缩放
        if temperature > 0:
            logits = logits / temperature

        # 概率
        probs = softmax(logits, dim=-1)

        # nucleus (top-p) 采样
        if 0 < top_p < 1.0:
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            # ... 见注意事项中的修正

        # 从分布中抽样一个 token
        next_token_id = torch.multinomial(probs, num_samples=1)

        # 早停：如果是特殊符号就停止
        if next_token_id.item() in tokenizer.special_tokens_encoder.values():
            break

        # 打印并把该 token 拼到序列末尾，继续下一步
        print(tokenizer.decode(next_token_id.tolist()[0]), end="", flush=True)
        ids = torch.cat((ids, next_token_id), dim=1)

    print("\n--- Generation finished ---")
```

* `@torch.no_grad()`：关闭梯度，推理必备。
* `model.eval()`：评估模式，关闭 dropout 等。
* **上下文 `context`**：每次只把**最近的一段**作为输入前向（解码阶段一 token 一步）。
* **logits 取最后一步**：`logits[:, -1, :]` 表示只关心“预测下一个 token”的那一刻的分布。
* **温度**：`logits /= T` 后 softmax；`T` 越大越平，越随机。
* **top-p (nucleus) 采样**：按概率从高到低排序，截取累计概率刚好超过 `p` 的最小集合，只在这个子集中归一化后采样，从而兼顾多样性与质量。&#x20;
* **采样与拼接**：`torch.multinomial` 从 `probs` 中抽一个 id；如果命中结束符就停止；否则解码打印并把它接到 `ids` 末尾。



# 3 `main()`：加载→复原→生成

```python
def main():
    args = get_args()
    # 设备
    device = args.device
    if device == "cuda" and not torch.cuda.is_available():
        print("CUDA not available, falling back to CPU")
        device = "cpu"
    torch.manual_seed(1337)

    # 分词器
    tokenizer = Tokenizer.from_files(
        vocab_filepath=args.vocab_path,
        merges_filepath=args.merges_path,
        special_tokens=args.special_tokens
    )

    # 加载 checkpoint
    checkpoint = torch.load(args.checkpoint_path, map_location=device)

    # 取出/推断模型结构参数，构建并加载权重
    model_args = checkpoint.get("model_args", None)
    if model_args is None:
        # 从 state_dict 维度推断（d_model, vocab_size, num_layers 等）
        ...
    model = TransformerLM(**model_args, device=device)
    model.load_state_dict(checkpoint['model_state'])
    model.to(device)

    # 可选编译加速
    if args.compile:
        model = torch.compile(model)

    # 编码提示并开始生成
    prompt_ids = tokenizer.encode(args.prompt)
    prompt = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
    generate(model, tokenizer, prompt, args.max_new_tokens, args.temperature, args.top_p)
```

* **设备选择与随机种子**：GPU 不可用则回退 CPU；设定随机种子保证复现实验。
* **分词器装载**：从 `vocab/merges` 文件构造 BPE 分词器实例。
* **恢复模型**：`torch.load` 读取 checkpoint，优先从 `checkpoint["model_args"]` 取配置；若缺失就从权重维度**推断**（`token_embeddings.weight.shape` 给出 `vocab_size` 与 `d_model`，根据层命名推断 `num_layers`），然后 `load_state_dict` 灌入权重。&#x20;
* **可选编译**：`torch.compile(model)` 在一些后端能带来推理速度提升。
* **编码提示并调用 `generate`**：把字符串 prompt 变成 token id 张量，扩一维成 `[B=1, T]`，开始循环解码。

# 4 演示

```bash
CHECKPOINT_PATH="./checkpoints/lr_sweep_3e-4/ckpt.pt"

uv run python generate.py \
    --checkpoint_path=$CHECKPOINT_PATH \
    --vocab_path=./bpe_tokenizer/tinystories_vocab.json \
    --merges_path=./bpe_tokenizer/tinystories_merges.txt \
    --prompt="Once upon a time, there was a brave knight named" \
    --max_new_tokens=150 \
    --temperature=0.8 \
    --top_p=0.9 \
    --device=cuda
```